# Connecting OpenAI to MCP Servers

This notebook demonstrates how to run a FastMCP server over HTTP and connect it to the OpenAI Responses API using the `type: "mcp"` tool, enabling GPT models to discover and call your MCP tools automatically.

## How OpenAI Connects to MCP

The OpenAI **Responses API** natively supports MCP as a tool type. When you pass an MCP server URL, OpenAI's runtime:

1. **Discovers tools** — calls the server's `tools/list` endpoint to learn what tools are available
2. **Decides to use tools** — the model decides which tools to call based on the user's input
3. **Executes tool calls** — OpenAI's runtime calls the tools on your server and feeds results back to the model

This means the model can use your custom tools without you manually defining JSON schemas — MCP handles that automatically.

### Requirements

- The MCP server must be **publicly accessible** over HTTP (OpenAI's servers need to reach it)
- We will use **ngrok** to create a public tunnel to our local server
- Only **tools** are supported by OpenAI (not resources or prompts — those are MCP features used by other clients)

## Setup

In [ ]:
%pip install -q fastmcp openai pyngrok

In [ ]:
import os
import json
import time
import threading
from getpass import getpass
from openai import OpenAI
from fastmcp import FastMCP

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4o"

## Part 1: Running an MCP Server over HTTP

In the previous notebook, we tested servers in-memory. To connect OpenAI, we need to run the server as an HTTP service.

FastMCP supports multiple transports:
- **stdio**: Local process communication (for Claude Desktop, etc.)
- **http**: Streamable HTTP (for remote access — what OpenAI needs)

Since `mcp.run()` is blocking, we run it in a **background thread** so we can continue using the notebook.

In [ ]:
import random

# Define our MCP server
mcp_server = FastMCP("Demo Tools")


@mcp_server.tool
def roll_dice(n_dice: int, sides: int = 6) -> dict:
    """Roll a specified number of dice. Returns individual rolls and the total."""
    rolls = [random.randint(1, sides) for _ in range(n_dice)]
    return {"rolls": rolls, "total": sum(rolls)}


@mcp_server.tool
def calculate_tip(bill_amount: float, tip_percentage: float = 18.0) -> dict:
    """Calculate the tip and total for a restaurant bill."""
    tip = bill_amount * (tip_percentage / 100)
    return {
        "bill": bill_amount,
        "tip_percentage": tip_percentage,
        "tip_amount": round(tip, 2),
        "total": round(bill_amount + tip, 2),
    }


@mcp_server.tool
def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    """Convert between common units. Supports: km/miles, kg/lbs, celsius/fahrenheit."""
    conversions = {
        ("km", "miles"): lambda v: v * 0.621371,
        ("miles", "km"): lambda v: v * 1.60934,
        ("kg", "lbs"): lambda v: v * 2.20462,
        ("lbs", "kg"): lambda v: v * 0.453592,
        ("celsius", "fahrenheit"): lambda v: (v * 9 / 5) + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key not in conversions:
        return {"error": f"Conversion from {from_unit} to {to_unit} not supported"}
    result = conversions[key](value)
    return {"input": f"{value} {from_unit}", "output": f"{round(result, 2)} {to_unit}"}


# Run server in a background thread
server_thread = threading.Thread(
    target=lambda: mcp_server.run(transport="http", host="127.0.0.1", port=8000),
    daemon=True,
)
server_thread.start()
time.sleep(2)  # Wait for server to start

print("MCP server running at http://127.0.0.1:8000/mcp/")
print("Tools available: roll_dice, calculate_tip, convert_units")

### Verify the Server Locally

Before exposing the server to OpenAI, let's verify it works using the FastMCP `Client` over HTTP.

In [ ]:
from fastmcp import Client

async with Client("http://127.0.0.1:8000/mcp/") as local_client:
    tools = await local_client.list_tools()
    print("Tools discovered via HTTP:")
    for tool in tools:
        print(f"  - {tool.name}: {tool.description}")
    print()

    result = await local_client.call_tool("roll_dice", {"n_dice": 3, "sides": 6})
    print(f"Roll 3d6: {result[0].text}")

## Part 2: Exposing the Server with ngrok

OpenAI's servers need to reach our MCP server over the public internet. [ngrok](https://ngrok.com/) creates a secure tunnel from a public URL to our local server.

### ngrok Setup

1. Sign up for a free account at [ngrok.com](https://ngrok.com/)
2. Get your auth token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Enter it when prompted below

In [ ]:
from pyngrok import ngrok

# Set your ngrok auth token (get it from https://dashboard.ngrok.com/get-started/your-authtoken)
if not os.environ.get("NGROK_AUTHTOKEN"):
    ngrok_token = getpass("Enter your ngrok auth token: ")
    ngrok.set_auth_token(ngrok_token)

# Create a tunnel to our local MCP server
public_url = ngrok.connect(8000)
mcp_public_url = f"{public_url}/mcp/"

print(f"Public MCP URL: {mcp_public_url}")
print("\nThis URL is now accessible from the internet (including OpenAI's servers).")

## Part 3: OpenAI Responses API + MCP

Now we can pass our MCP server to OpenAI. The key is the `type: "mcp"` tool configuration:

```python
{
    "type": "mcp",                      # Identifies this as an MCP tool
    "server_label": "my_server",         # Unique label for this server
    "server_url": "https://...",         # Public URL of your MCP server
    "require_approval": "never",         # Auto-approve tool calls
}
```

OpenAI's runtime will automatically:
1. Call `tools/list` on your server to discover available tools
2. Let the model decide which tools to use
3. Execute tool calls against your server
4. Feed results back to the model for a final response

In [ ]:
# Connect OpenAI to our MCP server
response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "demo_tools",
            "server_url": mcp_public_url,
            "require_approval": "never",
        }
    ],
    input="Roll 4 dice with 20 sides each.",
)

print(response.output_text)

In [ ]:
# Try the tip calculator
response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "demo_tools",
            "server_url": mcp_public_url,
            "require_approval": "never",
        }
    ],
    input="My dinner bill was $87.50. What should I tip at 20%?",
)

print(response.output_text)

In [ ]:
# Try the unit converter
response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "demo_tools",
            "server_url": mcp_public_url,
            "require_approval": "never",
        }
    ],
    input="Convert 42 kilometers to miles.",
)

print(response.output_text)

### Inspecting MCP Tool Calls

The response output contains detailed information about the MCP interaction. Let's inspect what happened behind the scenes.

In [ ]:
# Make a request and inspect the full output chain
response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "demo_tools",
            "server_url": mcp_public_url,
            "require_approval": "never",
        }
    ],
    input="I weigh 180 lbs — what is that in kg?",
)

print("=== Full Response Output ===")
for item in response.output:
    print(f"\nType: {item.type}")
    if item.type == "mcp_list_tools":
        print(f"  Server: {item.server_label}")
        print(f"  Tools discovered: {len(item.tools)}")
        for tool in item.tools:
            print(f"    - {tool.name}: {tool.description}")
    elif item.type == "mcp_tool_call":
        print(f"  Tool: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Server: {item.server_label}")
    elif item.type == "mcp_tool_call_output":
        print(f"  Output: {item.output}")
    elif item.type == "message":
        for content in item.content:
            if hasattr(content, "text"):
                print(f"  Text: {content.text}")

## Part 4: Multi-Turn Conversations with MCP

You can maintain conversation context across requests using `previous_response_id`. This also **caches the tool list** — OpenAI won't re-fetch tools from your server on follow-up requests, making them faster.

In [ ]:
mcp_tool = {
    "type": "mcp",
    "server_label": "demo_tools",
    "server_url": mcp_public_url,
    "require_approval": "never",
}

# First message
response1 = client.responses.create(
    model=MODEL,
    tools=[mcp_tool],
    input="Roll 2 six-sided dice for me.",
)
print(f"Turn 1: {response1.output_text}")
print()

# Follow-up message (uses previous_response_id to cache tool list)
response2 = client.responses.create(
    model=MODEL,
    tools=[mcp_tool],
    input="Now convert those total points from kilometers to miles.",
    previous_response_id=response1.id,
)
print(f"Turn 2: {response2.output_text}")

## Part 5: Combining MCP Tools with Built-in Tools

One of the most powerful features is combining your MCP server with OpenAI's built-in tools like `web_search_preview`. The model can use both in a single request.

In [ ]:
# Combine MCP tools with web search
response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "demo_tools",
            "server_url": mcp_public_url,
            "require_approval": "never",
        },
        {"type": "web_search_preview"},
    ],
    input="What is the distance from London to Paris in kilometers? Then convert that to miles using the convert_units tool.",
)

print(response.output_text)

## Part 6: Filtering Tools with `allowed_tools`

If your MCP server exposes many tools but you only want the model to use specific ones, use the `allowed_tools` parameter.

In [ ]:
# Only expose the roll_dice tool — the model cannot use calculate_tip or convert_units
response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "demo_tools",
            "server_url": mcp_public_url,
            "require_approval": "never",
            "allowed_tools": ["roll_dice"],
        }
    ],
    input="Roll 5 dice with 10 sides.",
)

print(response.output_text)

## Cleanup

Disconnect the ngrok tunnel when done.

In [ ]:
ngrok.disconnect(public_url.public_url)
print("ngrok tunnel closed.")

## Summary

- **FastMCP servers can run over HTTP** using `mcp.run(transport="http")`, making them accessible to remote clients.
- **ngrok** creates a public tunnel to your local server so OpenAI can reach it.
- The OpenAI Responses API supports MCP via `{"type": "mcp", "server_url": "..."}` — OpenAI automatically discovers and calls your tools.
- **`previous_response_id`** enables multi-turn conversations and caches the tool list for faster follow-up requests.
- You can **combine MCP tools with built-in tools** like `web_search_preview` in a single request.
- **`allowed_tools`** lets you filter which MCP tools the model can access.
- Only **tools** are supported by OpenAI's MCP integration (not resources or prompts).